**Imports and paths set up**

In [19]:
from pathlib import Path
import pandas as pd

In [20]:
PROJECT_ROOT = Path("./")
NPI_DATA_DIR = PROJECT_ROOT / "npi-dataset"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
WEBPAGE_DATA_DIR = PROJECT_ROOT / "docs" / "data"

Load the .csv npi dataset files.
- anzsic-2006.csv
- reports.csv
- emission-reduction-techniques.csv
- emissions.csv
- facilities.csv
- substances.csv
- data-dictionary.csv

In [21]:
# Task 1: handle the header BOM. Task 2: load as strings before parsing values.
# This preserves leading zeros in identifiers, ANZSIC codes and postcodes.
anzsic_2006_dataset = pd.read_csv(NPI_DATA_DIR / "anzsic-2006.csv", encoding="utf-8-sig", dtype="string")

reports_dataset = pd.read_csv(NPI_DATA_DIR / "reports.csv", encoding="utf-8-sig", dtype="string")

emission_reduction_techniques_dataset = pd.read_csv(NPI_DATA_DIR / "emission-reduction-techniques.csv", encoding="utf-8-sig", dtype="string")

emissions_dataset = pd.read_csv(NPI_DATA_DIR / "emissions.csv", encoding="utf-8-sig", dtype="string")

facilities_dataset = pd.read_csv(NPI_DATA_DIR / "facilities.csv", encoding="utf-8-sig", dtype="string")

substances_dataset = pd.read_csv(NPI_DATA_DIR / "substances.csv", encoding="utf-8-sig", dtype="string")

data_dictionary_dataset = pd.read_csv(NPI_DATA_DIR / "data-dictionary.csv", encoding="utf-8-sig", dtype="string")

In [22]:
# Task 2: parse emission amounts and dates separately from identifiers.
# Missing amounts stay missing. Keep raw values and flag malformed/negative amounts.
emission_amount_columns = [
    "air_point_emission_kg",
    "air_fugitive_emission_kg",
    "air_total_emission_kg",
    "water_emission_kg",
    "land_emission_kg",
]
for column in emission_amount_columns:
    emissions_dataset[column + "_raw"] = emissions_dataset[column]
    emissions_dataset[column] = pd.to_numeric(emissions_dataset[column], errors="coerce")
    emissions_dataset[column + "_non_numeric"] = (
        emissions_dataset[column + "_raw"].notna() & emissions_dataset[column].isna()
    )
    emissions_dataset[column + "_negative"] = emissions_dataset[column].lt(0).fillna(False)

report_date_columns = ["data_start", "data_end", "first_published", "last_updated"]
for column in report_date_columns:
    reports_dataset[column + "_raw"] = reports_dataset[column]
    reports_dataset[column] = pd.to_datetime(
        reports_dataset[column], format="%Y-%m-%d", errors="coerce"
    )
    reports_dataset[column + "_invalid"] = (
        reports_dataset[column + "_raw"].notna() & reports_dataset[column].isna()
    )

In [23]:
# Task 3: classify each report using its primary ANZSIC class at that time.
manufacturing_anzsic_dataset = anzsic_2006_dataset.loc[
    anzsic_2006_dataset["division_code"].eq("C")].copy() # "C" is Manufacturing division from anzsic-2006.csv

manufacturing_reports_dataset = reports_dataset.merge(
    manufacturing_anzsic_dataset[
        ["class_code", "class_name", "division_code", "division_name",
         "subdivision_code", "subdivision_name", "group_code", "group_name"]
    ],
    left_on="primary_anzsic_class_code",
    right_on="class_code",
    how="inner",
    validate="many_to_one",
) # Join the columns. All reports filtered reports to manufacturing_reports_dataset

# Retain rows by report ID, not by a facility's current industry classification.
# Keep techniques and emissions separate to avoid multiplying emission amounts.
manufacturing_report_ids = manufacturing_reports_dataset["report_id"]
manufacturing_techniques_dataset = emission_reduction_techniques_dataset.loc[
    emission_reduction_techniques_dataset["report_id"].isin(manufacturing_report_ids)
].copy()
manufacturing_emissions_dataset = emissions_dataset.loc[
    emissions_dataset["report_id"].isin(manufacturing_report_ids)
].copy()

pd.Series({
    "Manufacturing classes": len(manufacturing_anzsic_dataset),
    "Manufacturing facilities with reports": manufacturing_reports_dataset["facility_id"].nunique(),
    "Manufacturing reports": len(manufacturing_reports_dataset),
    "Manufacturing technique rows": len(manufacturing_techniques_dataset),
    "Manufacturing emissions rows": len(manufacturing_emissions_dataset),
}, name="Count")

Manufacturing classes                       143
Manufacturing facilities with reports      2364
Manufacturing reports                     33265
Manufacturing technique rows             126541
Manufacturing emissions rows             335844
Name: Count, dtype: int64

### Facility identity and estimation methods
Use `facility_id` across years and retain the facility lookup for facilities with at least one manufacturing report. 

The lookup contains current information. historical industry membership comes from reports. 

Emissions and techniques remain separate. Keep all EET fields unchanged, including `air_point_eet` and `air_fugitive_eet`.

In [24]:
manufacturing_facilities_dataset = facilities_dataset.loc[
    facilities_dataset["facility_id"].isin(manufacturing_reports_dataset["facility_id"])
].copy() #check facilities are located in the manufacturing_reports_dataset, at least 1 report mentioned for any facility 

# air_point_eet, air_fugitive_eet, water_eet, land_eet. I want these cols only.
emission_estimation_columns = [
    "air_point_eet",
    "air_fugitive_eet",
    "water_eet",
    "land_eet",
]
assert {"air_point_eet", "air_fugitive_eet"}.issubset(emission_estimation_columns) #Validation check
assert set(emission_estimation_columns).issubset(manufacturing_emissions_dataset.columns) #Validation check

# Inclusive dates: 1 July to 30 June represents 365 or 366 days.

# Calculating Duration. Subtracts the start date from the end date to get the total duration
manufacturing_reports_dataset["period_days"] = (
    manufacturing_reports_dataset["data_end"] - manufacturing_reports_dataset["data_start"]
).dt.days + 1

# Identifying Issues & Flagging Missing/Invalid Dates
manufacturing_reports_dataset["period_dates_missing"] = (
    manufacturing_reports_dataset[["data_start", "data_end"]].isna().any(axis=1)
)
manufacturing_reports_dataset["period_dates_invalid"] = (
    manufacturing_reports_dataset[["data_start_invalid", "data_end_invalid"]].any(axis=1)
)

# Flags True if the end date accidentally comes before the start date (meaning period_days is less than or equal to 0).
manufacturing_reports_dataset["period_reversed"] = manufacturing_reports_dataset["period_days"].le(0).fillna(False)

#Flags True if the reporting duration is exactly a standard year (365 days) or a leap year (366 days).
manufacturing_reports_dataset["period_full_year"] = manufacturing_reports_dataset["period_days"].isin([365, 366])

#Flags True if the report covers a short or partial period lasting anywhere from 1 day up to 364 days.
manufacturing_reports_dataset["period_partial"] = manufacturing_reports_dataset["period_days"].between(1, 364).fillna(False)

# Flags rows that have a weird or irregular timeframe. It looks for records where the dates are not missing (~), 
# but they are not a standard full year (~). This will catch reversed dates, abnormally long periods (over 366 days), 
# or any duration that doesn't fit standard reporting constraints.
manufacturing_reports_dataset["period_nonstandard_length"] = (
    ~manufacturing_reports_dataset["period_dates_missing"]
    & ~manufacturing_reports_dataset["period_full_year"]
)
# Length alone does not establish a consistent annual calendar; later windows check it.

### Validate keys, relationships and values
Structural failures stop export; records  never silently deduplicated or dropped. 

Value/period issues retained with flags and counted for later eligibility decisions. `air_total_emission_kg` is used directly; 

this notebook calculates no emission totals across pathways or substances.

In [25]:
quality_rows = []

def record_check(table, check, affected_rows, severity="error", detail=""):
    quality_rows.append({
        "table": table, "check": check, "affected_rows": int(affected_rows),
        "severity": severity,
        "status": "pass" if affected_rows == 0 else ("fail" if severity == "error" else "flagged"),
        "detail": detail,
    })

# Check the entire source snapshot before trusting its manufacturing subset.
key_checks = [
    ("anzsic", anzsic_2006_dataset, ["class_code"]),
    ("reports", reports_dataset, ["report_id"]),
    ("reports", reports_dataset, ["facility_id", "report_year"]),
    ("facilities", facilities_dataset, ["facility_id"]),
    ("substances", substances_dataset, ["substance_id"]),
    ("emissions", emissions_dataset, ["report_id", "substance_id"]),
    ("techniques", emission_reduction_techniques_dataset, ["report_id", "reduction_technique_id"]),
]
for name, dataset, keys in key_checks:
    missing = dataset[keys].isna().any(axis=1) | dataset[keys].eq("").any(axis=1)
    record_check(name, "missing_key: " + ", ".join(keys), missing.sum())
    record_check(name, "duplicate_key: " + ", ".join(keys), dataset.duplicated(keys, keep=False).sum())

record_check("reports", "missing_facility_id", reports_dataset["facility_id"].isna().sum())
record_check("reports", "missing_primary_class", reports_dataset["primary_anzsic_class_code"].isna().sum())
record_check("reports", "unmatched_primary_class", (~reports_dataset["primary_anzsic_class_code"].isin(anzsic_2006_dataset["class_code"])).sum())
record_check("reports", "unmatched_facility", (~reports_dataset["facility_id"].isin(facilities_dataset["facility_id"])).sum())
record_check("emissions", "unmatched_substance", (~emissions_dataset["substance_id"].isin(substances_dataset["substance_id"])).sum())

# Joining only report metadata validates links without combining techniques and substances.
for name, dataset in [("emissions", emissions_dataset), ("techniques", emission_reduction_techniques_dataset)]:
    linked = dataset[["report_id", "facility_id", "report_year"]].merge(
        reports_dataset[["report_id", "facility_id", "report_year"]],
        on="report_id", how="left", suffixes=("_child", "_report"),
        validate="many_to_one", indicator=True,
    )
    record_check(name, "unmatched_report", linked["_merge"].ne("both").sum())
    for column in ["facility_id", "report_year"]:
        mismatch = linked[column + "_child"].isna() | linked[column + "_child"].ne(linked[column + "_report"]).fillna(True)
        record_check(name, column + "_differs_from_report", mismatch.sum())

for column in report_date_columns:
    record_check("reports", column + "_invalid", reports_dataset[column + "_invalid"].sum(), "warning")
for column in ["period_dates_missing", "period_dates_invalid", "period_reversed", "period_partial", "period_nonstandard_length"]:
    record_check("manufacturing_reports", column, manufacturing_reports_dataset[column].sum(), "warning", "Retained with period flags; assess before selecting annual windows.")
for name, dataset in [("emissions", emissions_dataset), ("manufacturing_emissions", manufacturing_emissions_dataset)]:
    for column in emission_amount_columns:
        for flag in ["non_numeric", "negative"]:
            record_check(name, column + "_" + flag, dataset[column + "_" + flag].sum(), "warning")
        record_check(name, column + "_missing", dataset[column].isna().sum(), "info", "Missing is not zero; non-numeric raw values are retained separately.")

# Verify filtering lost no eligible rows and preserved original EET values.
for name, original, subset in [
    ("manufacturing_emissions", emissions_dataset, manufacturing_emissions_dataset),
    ("manufacturing_techniques", emission_reduction_techniques_dataset, manufacturing_techniques_dataset),
]:
    expected = original.loc[original["report_id"].isin(manufacturing_report_ids)]
    pd.testing.assert_frame_equal(subset, expected)
    record_check(name, "subset_matches_report_membership", 0)
record_check("manufacturing_facilities", "missing_lookup", (~manufacturing_reports_dataset["facility_id"].isin(manufacturing_facilities_dataset["facility_id"])).sum())

for name, dataset in [("reports", manufacturing_reports_dataset), ("techniques", manufacturing_techniques_dataset), ("emissions", manufacturing_emissions_dataset), ("facilities", manufacturing_facilities_dataset)]:
    record_check("manufacturing_" + name, "row_count", 0, "info", str(len(dataset)))

data_quality_summary = pd.DataFrame(quality_rows)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
data_quality_summary.to_csv(RESULTS_DIR / "data_quality_summary.csv", index=False)
structural_failures = data_quality_summary.query("severity == 'error' and status == 'fail'")
assert structural_failures.empty, "Investigate structural failures in data_quality_summary.csv before export."
data_quality_summary.loc[data_quality_summary["status"].ne("pass")]

,table,check,affected_rows,severity,status,detail
29,manufacturing_reports,period_dates_missing,1,warning,flagged,Retained with period flags; assess before sele...
32,manufacturing_reports,period_partial,93,warning,flagged,Retained with period flags; assess before sele...
33,manufacturing_reports,period_nonstandard_length,96,warning,flagged,Retained with period flags; assess before sele...
36,emissions,air_point_emission_kg_missing,451554,info,flagged,Missing is not zero; non-numeric raw values ar...
39,emissions,air_fugitive_emission_kg_missing,412539,info,flagged,Missing is not zero; non-numeric raw values ar...
42,emissions,air_total_emission_kg_missing,74426,info,flagged,Missing is not zero; non-numeric raw values ar...
45,emissions,water_emission_kg_missing,929097,info,flagged,Missing is not zero; non-numeric raw values ar...
48,emissions,land_emission_kg_missing,879568,info,flagged,Missing is not zero; non-numeric raw values ar...
51,manufacturing_emissions,air_point_emission_kg_missing,43320,info,flagged,Missing is not zero; non-numeric raw values ar...
54,manufacturing_emissions,air_fugitive_emission_kg_missing,182688,info,flagged,Missing is not zero; non-numeric raw values ar...


### Save prepared tables

Save the four manufacturing tables as local Parquet files and read them back to verify values and types. Source CSVs remain unchanged.

In [26]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

prepared_datasets = {
    "manufacturing_reports": manufacturing_reports_dataset,
    "manufacturing_techniques": manufacturing_techniques_dataset,
    "manufacturing_emissions": manufacturing_emissions_dataset,
    "manufacturing_facilities": manufacturing_facilities_dataset,
}

for name, dataset in prepared_datasets.items():
    output_path = PROCESSED_DIR / (name + ".parquet")
    dataset.to_parquet(output_path, index=False)
    # Check persisted values and types, including missing values and EET strings.
    pd.testing.assert_frame_equal(dataset.reset_index(drop=True), pd.read_parquet(output_path))

pd.DataFrame([{"table": name, "rows": len(dataset), "columns": len(dataset.columns)}
              for name, dataset in prepared_datasets.items()])

,table,rows,columns
0,manufacturing_reports,33265,47
1,manufacturing_techniques,126541,14
2,manufacturing_emissions,335844,42
3,manufacturing_facilities,2364,22


### Section 5 checks and outputs
- [x] Required keys checked for missing values and duplicates; structural failures stop export.
- [x] ANZSIC mapping is many-to-one; child tables link to one report, with facility/year consistency checked.
- [x] Report dates parsed, inclusive durations calculated, missing/invalid/partial/nonstandard periods flagged.
- [x] Non-numeric and negative emissions flagged; original amounts retained and missing values preserved.
- [x] Total air emissions retained directly, alongside point/fugitive amounts and EET fields.
- [x] No totals across substances and no combined technique–emissions join.
- [x] Four manufacturing Parquet tables saved under `data/processed/` and read back for verification.
- [x] `data_quality_summary.csv` saved under `data/results/`.

A completed check means validation ran, not that every report has a complete annual period or every emission is observed. Inspect flagged quality-summary rows before later analyses.